In [ ]:
import os
import time
import torch
import torch.nn as nn
from torchvision import models
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ─────────────────────────────────────────
# 0. SETUP
# ─────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/complexity_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
DUMMY_INPUT = torch.randn(1, 3, 224, 224).to(DEVICE)
# InceptionV3 requires 299x299 input (not 224x224)
DUMMY_INPUT_INCEPTION = torch.randn(1, 3, 299, 299).to(DEVICE)

print("Device:", DEVICE)

# ─────────────────────────────────────────
# 1. BUILDING BLOCKS (same as your model)
# ─────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean(dim=(2,3))
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_p = x.mean(dim=(2,3))
        max_p = torch.amax(x, dim=(2,3))
        ca    = torch.sigmoid(self.mlp(avg_p) + self.mlp(max_p))
        x     = x * ca.view(x.size(0), x.size(1), 1, 1)
        sa    = torch.cat([x.mean(dim=1,keepdim=True),
                           torch.amax(x,dim=1,keepdim=True)], dim=1)
        sa    = self.sigmoid(self.conv(sa))
        return x * sa


def unfreeze_last_n(model, n):
    for p in model.parameters():
        p.requires_grad = False
    for p in list(model.parameters())[-n:]:
        p.requires_grad = True


# ─────────────────────────────────────────
# 2. ALL MODELS
# ─────────────────────────────────────────

# ── Baseline 1: DenseNet121 ──
class DenseNet121Baseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        in_f = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Linear(in_f, num_classes)
    def forward(self, x):
        return self.backbone(x)


# ── Baseline 2: EfficientNet-B0 ──
class EfficientNetB0Baseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Linear(in_f, num_classes)
    def forward(self, x):
        return self.backbone(x)


# ── Baseline 3: InceptionV3 (FIXED: aux_logits must stay True when loading
#    pretrained weights; we swap in our own fc + AuxLogits.fc afterwards,
#    and the forward pass returns only the main logits in eval mode) ──
class InceptionV3Baseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.inception_v3(
            weights=models.Inception_V3_Weights.IMAGENET1K_V1
        )
        in_f = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_f, num_classes)
        self.backbone.AuxLogits.fc = nn.Linear(768, num_classes)

    def forward(self, x):
        out = self.backbone(x)
        # InceptionV3 returns InceptionOutputs(logits, aux) in train mode,
        # and just the tensor in eval mode.
        if isinstance(out, tuple):
            return out[0]
        return out


# ── Baseline 4: MobileNetV2 ──
class MobileNetV2Baseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        in_f = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Linear(in_f, num_classes)
    def forward(self, x):
        return self.backbone(x)


# ── Baseline 5: ResNet50 ──
class ResNet50Baseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        in_f = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_f, num_classes)
    def forward(self, x):
        return self.backbone(x)


# ── Baseline 6: Swin Transformer ──
class SwinTBaseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
        in_f = self.backbone.head.in_features
        self.backbone.head = nn.Linear(in_f, num_classes)
    def forward(self, x):
        return self.backbone(x)


# ── Baseline 7: ViT-B/16 ──
class ViTBaseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        in_f = self.backbone.heads.head.in_features
        self.backbone.heads.head = nn.Linear(in_f, num_classes)
    def forward(self, x):
        return self.backbone(x)


# ── Proposed: CXRNet ──
class CXRNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.densenet = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.densenet.classifier = nn.Identity()
        unfreeze_last_n(self.densenet, 50)
        self.se_d = SEBlock(1024)

        self.efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.efficientnet.classifier = nn.Identity()
        for p in self.efficientnet.parameters():
            p.requires_grad = False
        self.se_e = SEBlock(1280)

        self.head_d = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.head_e = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.cbam = CBAM(512)
        self.classifier = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        fd = self.densenet(x).unsqueeze(-1).unsqueeze(-1)
        fd = self.se_d(fd).squeeze(-1).squeeze(-1)
        fd = self.head_d(fd)

        fe = self.efficientnet(x).unsqueeze(-1).unsqueeze(-1)
        fe = self.se_e(fe).squeeze(-1).squeeze(-1)
        fe = self.head_e(fe)

        f  = torch.cat([fd, fe], dim=1).unsqueeze(-1).unsqueeze(-1)
        f  = self.cbam(f).squeeze(-1).squeeze(-1)
        return self.classifier(f)


# ─────────────────────────────────────────
# 3. COMPLEXITY MEASUREMENT FUNCTIONS
# ─────────────────────────────────────────

def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def get_model_size_mb(model):
    """Size of model weights in MB."""
    total_bytes = sum(
        p.numel() * p.element_size() for p in model.parameters()
    )
    return total_bytes / (1024 ** 2)


def get_flops_manual(model, input_tensor):
    """
    GFLOPs using torch.profiler — no external library needed.
    Falls back to None if profiler fails.
    """
    try:
        from torch.profiler import profile as torch_profile, ProfilerActivity
        model.eval()
        with torch_profile(
            activities=[ProfilerActivity.CPU],
            record_shapes=True,
            with_flops=True
        ) as prof:
            with torch.no_grad():
                model(input_tensor)
        total_flops = sum(
            e.flops for e in prof.key_averages() if e.flops is not None
        )
        return total_flops / 1e9  # GFLOPs
    except Exception:
        return None


def get_inference_time_ms(model, input_tensor, runs=50, warmup=10):
    """
    Average inference time in ms over `runs` forward passes.
    Uses CUDA events if GPU available, else time.time().
    """
    model.eval()
    use_cuda = (input_tensor.device.type == "cuda")

    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_tensor)

    if use_cuda:
        torch.cuda.synchronize()
        starter = torch.cuda.Event(enable_timing=True)
        ender   = torch.cuda.Event(enable_timing=True)
        times   = []
        with torch.no_grad():
            for _ in range(runs):
                starter.record()
                _ = model(input_tensor)
                ender.record()
                torch.cuda.synchronize()
                times.append(starter.elapsed_time(ender))
        return float(np.mean(times))
    else:
        times = []
        with torch.no_grad():
            for _ in range(runs):
                t0 = time.perf_counter()
                _ = model(input_tensor)
                times.append((time.perf_counter() - t0) * 1000)
        return float(np.mean(times))


# ─────────────────────────────────────────
# 4. RUN COMPLEXITY ANALYSIS
# ─────────────────────────────────────────
model_registry = {
    "DenseNet121"       : DenseNet121Baseline(NUM_CLASSES),
    "EfficientNet-B0"   : EfficientNetB0Baseline(NUM_CLASSES),
    "InceptionV3"       : InceptionV3Baseline(NUM_CLASSES),
    "MobileNetV2"       : MobileNetV2Baseline(NUM_CLASSES),
    "ResNet50"          : ResNet50Baseline(NUM_CLASSES),
    "Swin Transformer"  : SwinTBaseline(NUM_CLASSES),
    "ViT-B/16"          : ViTBaseline(NUM_CLASSES),
    "CXRNet (Proposed)" : CXRNet(NUM_CLASSES),
}

rows = []
for name, model in model_registry.items():
    print(f"\nAnalyzing {name} ...")
    model = model.to(DEVICE)
    model.eval()

    # InceptionV3 needs 299x299 input; everything else uses 224x224
    inp = DUMMY_INPUT_INCEPTION if name == "InceptionV3" else DUMMY_INPUT

    total_p, trainable_p = count_params(model)
    size_mb   = get_model_size_mb(model)
    gflops    = get_flops_manual(model, inp)
    infer_ms  = get_inference_time_ms(model, inp)

    rows.append({
        "Model"               : name,
        "Total Params (M)"    : round(total_p / 1e6, 2),
        "Trainable Params (M)": round(trainable_p / 1e6, 2),
        "Model Size (MB)"     : round(size_mb, 2),
        "GFLOPs"               : round(gflops, 2) if gflops is not None else "N/A",
        "Inference Time (ms)" : round(infer_ms, 2),
    })

    # free GPU memory between models
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ─────────────────────────────────────────
# 5. SAVE TABLE
# ─────────────────────────────────────────
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUTPUT_DIR, "model_complexity.csv"), index=False)

print("\n\n========== MODEL COMPLEXITY TABLE ==========")
print(df.to_string(index=False))

# ─────────────────────────────────────────
# 6. PLOTS
# ─────────────────────────────────────────
model_names = df["Model"].tolist()
colors      = ["#4C72B0"] * (len(model_names)-1) + ["#DD8452"]  # last = proposed (orange)
x           = np.arange(len(model_names))

# ── Plot 1: Total Params ──
plt.figure(figsize=(11, 5))
bars = plt.bar(model_names, df["Total Params (M)"], color=colors, edgecolor="black")
for bar, val in zip(bars, df["Total Params (M)"]):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f"{val}M", ha="center", va="bottom", fontsize=9)
plt.ylabel("Total Parameters (M)")
plt.title("Model Parameter Count Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "params_comparison.png"), dpi=300, bbox_inches="tight")
plt.close()

# ── Plot 2: Inference Time ──
plt.figure(figsize=(11, 5))
bars = plt.bar(model_names, df["Inference Time (ms)"], color=colors, edgecolor="black")
for bar, val in zip(bars, df["Inference Time (ms)"]):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             f"{val}ms", ha="center", va="bottom", fontsize=9)
plt.ylabel("Inference Time (ms)")
plt.title("Inference Time Comparison (1 Image, GPU)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "inference_time_comparison.png"), dpi=300, bbox_inches="tight")
plt.close()

# ── Plot 3: Model Size ──
plt.figure(figsize=(11, 5))
bars = plt.bar(model_names, df["Model Size (MB)"], color=colors, edgecolor="black")
for bar, val in zip(bars, df["Model Size (MB)"]):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f"{val}MB", ha="center", va="bottom", fontsize=9)
plt.ylabel("Model Size (MB)")
plt.title("Model Size Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_size_comparison.png"), dpi=300, bbox_inches="tight")
plt.close()

# ── Plot 4: GFLOPs (skip if all N/A) ──
gflop_vals = df["GFLOPs"].tolist()
if any(v != "N/A" for v in gflop_vals):
    safe_vals = [v if v != "N/A" else 0 for v in gflop_vals]
    plt.figure(figsize=(11, 5))
    bars = plt.bar(model_names, safe_vals, color=colors, edgecolor="black")
    for bar, val in zip(bars, safe_vals):
        plt.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.05,
                 f"{val:.2f}", ha="center", va="bottom", fontsize=9)
    plt.ylabel("GFLOPs")
    plt.title("Computational Cost (GFLOPs)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "gflops_comparison.png"), dpi=300, bbox_inches="tight")
    plt.close()

# ── Plot 5: Grouped — Params + Size + Inference ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics_plot = [
    ("Total Params (M)",    "Parameters (M)",  "Total Parameters"),
    ("Model Size (MB)",     "Size (MB)",        "Model Size"),
    ("Inference Time (ms)", "Time (ms)",        "Inference Time"),
]
for ax, (col, ylabel, title) in zip(axes, metrics_plot):
    vals = df[col].tolist()
    b    = ax.bar(model_names, vals, color=colors, edgecolor="black")
    for bar, val in zip(b, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height(),
                f"{val}", ha="center", va="bottom", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=25, ha="right", fontsize=8)

plt.suptitle("Model Complexity Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "complexity_grouped.png"), dpi=300, bbox_inches="tight")
plt.close()

print(f"\n✅ All complexity results saved to: {OUTPUT_DIR}")